In [ ]:
### Python 3.10.13
# ── Future compatibility ───────────────────────────────────────────────────────
from __future__ import annotations 

# ── Basic libraries ──────────────────────────────────────────────
import numpy as np    # numerical arrays and math operations
import pandas as pd   # tabular data manipulation (DataFrames)


# ── Formatting libraries ──────────────────────────────────────────────────────────
import asyncio      
import json           
import os             
import pickle        
import re           
import sys            
from typing import (  
    Any,
    Dict,
    List,
    Literal,
    Optional,
    Type,
)

# ── AI / LLM ─────────────────────────────────────────────────────
from openai import AsyncOpenAI                          # async OpenAI API client
from pydantic import BaseModel, Field, create_model     # data validation and schema definition
from pydantic_ai import Agent                           # high-level LLM agent abstraction
from pydantic_ai.models.openai import OpenAIChatModel   # pydantic-ai wrapper for OpenAI chat models
from pydantic_ai.providers.openai import OpenAIProvider # provider config (base URL, auth) for OpenAI
from enum import Enum

# ── LLM API Credits ─────────────────────────────────────────────────────────────────────
sys.path.append('/Users/williamharrigan/Desktop/test_wagner')
import creds 

# ── Script containing Wagner Functions ─────────────────────────────────────────────────────────────────────
from wagner_functions import *

In [ ]:
# ----------------------------
# OpenAI client + model
# ----------------------------
openai_client = AsyncOpenAI(
    api_key=os.environ["OPENAI_API_KEY"]
)

model = OpenAIChatModel(
    "gpt-5.4", ## GPT 5.4 - balances reasoning and efficiency
    provider=OpenAIProvider(openai_client=openai_client),
)

In [ ]:
## Read in OCR text of Wagner passages

# text_inputs = {'GENUS_SPECIES': 'Family: 
# Genus: 
# Species: 
# OCR text from Wagner passages', ...}

text_inputs = parse_text_inputs("wagner_species.txt")

In [ ]:
# ----------------------------
# Enums
# ----------------------------
class Description(str, Enum):
    DICOTS = "Dicots"
    MONOCOTS = "Monocots"
    CONIFERS = "Conifers"
    FERNS = "Ferns and fern allies"

class LifeFormType(str, Enum):
    ANNUAL_HERB = "ANNUAL_HERB"
    BIENNIAL_HERB = "BIENNIAL_HERB"
    PERENNIAL_HERB = "PERENNIAL_HERB"
    EPIPHYTE = "EPIPHYTE"
    VINE = "VINE"
    SHRUB = "SHRUB"
    TREE = "TREE"
    SHRUB_TREE = "SHRUB/TREE"
    GRASS = "GRASS"
    SEDGE = "SEDGE"


class StemHairType(str, Enum):
    DENDRITIC = "DENDRITIC"
    GLABROUS = "GLABROUS"
    HIRSUTE = "HIRSUTE"
    HISPID = "HISPID"
    LEPIDOTE = "LEPIDOTE"
    PILOSE = "PILOSE"
    PUBERULENT = "PUBERULENT"
    STRIGOSE = "STRIGOSE"
    STELLATE = "STELLATE"
    TOMENTOSE = "TOMENTOSE"
    VILLOUS = "VILLOUS"
    GLAUCOUS = "GLAUCOUS"

class LeafType(str, Enum):
    SIMPLE = "SIMPLE"
    COMPOUND = "COMPOUND"


class LeafShapeType(str, Enum):
    ACEROSE = "ACEROSE"
    AWL_SHAPED = "AWL_SHAPED"
    GLADIATE = "GLADIATE"
    HASTATE = "HASTATE"
    CORDATE = "CORDATE"
    DELTOID = "DELTOID"
    # LANCEOLATE = "LANCEOLATE"
    # LINEAR = "LINEAR"
    ELLIPTIC = "ELLIPTIC"
    ENSIFORM = "ENSIFORM"
    LYRATE = "LYRATE"
    OBCORDATE = "OBCORDATE"
    FALCATE = "FALCATE"
    FLABELLATE = "FLABELLATE"
    OBDELTOID = "OBDELTOID"
    OBELLIPTIC = "OBELLIPTIC"
    OBLANCEOLATE = "OBLANCEOLATE"
    OBLONG = "OBLONG"
    PERFOLIATE = "PERFOLIATE"
    QUADRATE = "QUADRATE"
    OBOVATE = "OBOVATE"
    ORBICULAR = "ORBICULAR"
    RENIFORM = "RENIFORM"
    RHOMBIC = "RHOMBIC"
    OVAL = "OVAL"
    OVATE = "OVATE"
    ROTUND = "ROTUND"
    SAGITTATE = "SAGITTATE"
    PANDURATE = "PANDURATE"
    PELTATE = "PELTATE"
    SPATULATE = "SPATULATE"
    SUBULATE = "SUBULATE"
    CUNEATE = "CUNEATE"


class LeafMarginType(str, Enum):
    TEETH = "TEETH"
    LOBED = "LOBED"
    ENTIRE = "ENTIRE"
    NOTEETH = "NO TEETH"


class PhyllotaxyType(str, Enum):
    ALTERNATE = "ALTERNATE"
    OPPOSITE = "OPPOSITE"
    WHORLED = "WHORLED"
    DECUSSATE = "DECUSSATE"
    DISTICHOUS = "DISTICHOUS"
    # EQUITANT = "EQUITANT"
    # TERNATE = "TERNATE"
    # CAULINE = "CAULINE"

class LeafHairType(str, Enum):
    DENDRITIC = "DENDRITIC"
    GLABROUS = "GLABROUS"
    HIRSUTE = "HIRSUTE"
    HISPID = "HISPID"
    LEPIDOTE = "LEPIDOTE"
    PILOSE = "PILOSE"
    PUBERULENT = "PUBERULENT"
    STRIGOSE = "STRIGOSE"
    STELLATE = "STELLATE"
    TOMENTOSE = "TOMENTOSE"
    VILLOUS = "VILLOUS"
    GLAUCOUS = "GLAUCOUS"

class InflorescenceType(str, Enum):
    AXILLARY = "AXILLARY"
    CATKIN = "CATKIN"
    CYME = "CYME"
    CLUSTERS = "CLUSTERS"
    DICHASIUM = "DICHASIUM"
    FASCICLE = "FASCICLE"
    GLOMERATE = "GLOMERATE"
    HEAD = "HEAD"
    PANICLE = "PANICLE"
    RACEME = "RACEME"
    SPATHE_SPADIX = "SPATHE_SPADIX"
    THYRSE = "THYRSE"
    UMBEL = "UMBEL"
    VERTISCILLATE = "VERTISCILLATE"
    SOLITARY = "SOLITARY"
    SPIKE = "SPIKE"
    SPADIX = "SPADIX"


class CorollaType(str, Enum):
    TUBULAR = "TUBULAR"
    CAMPANULATE = "CAMPANULATE"
    FUNNELFORM = "FUNNELFORM"
    ROTATE = "ROTATE"
    SALVERFORM = "SALVERFORM"
    BILABIATE = "BILABIATE"
    ZYGOMORPHIC = "ZYGOMORPHIC"
    URCEOLATE = "URCEOLATE"
    CUP = "CUP"
    OBOVATE = "OBOVATE"
    ELIPTIC = "ELLIPTIC"
    OBLONG = "OBLONG"
    ORBICULAR = "ORBICULAR"
    


class BreedingType(str, Enum):
    MONOECIOUS = "MONOECIOUS"
    DIOECIOUS = "DIOECIOUS"
    POLYGAMOUS = "POLYGAMOUS"
    ANDROMONOECIOUS = "ANDROMONOECIOUS"
    GYNODIOECIOUS = "GYNODIOECIOUS"
    CLEISTOGAMOUS = "CLEISTOGAMOUS"


class Location(str, Enum):
    HAWAII = "HAWAII"
    MAUI = "MAUI"
    KAHOOLAWE = "KAHOOLAWE"
    MOLOKAI = "MOLOKAI"
    LANAI = "LANAI"
    OAHU = "OAHU"
    KAUAI = "KAUAI"
    NIIHAU = "NIIHAU"
    ALL_ISLANDS = "ALL ISLANDS"


class OriginType(str, Enum):
    INDIGENOUS = "INDIGENOUS"
    ENDEMIC = "ENDEMIC"
    PC = "PALEOTROPICAL COSMOPOLITAN - Originating from the paleotropics"
    NATURALIZED = "NATURALIZED"

class StatusType(str, Enum):
    NATURALIZED = "NATURALIZED"
    ENDEMIC = "ENDEMIC"
    RARE = "RARE"
    SECURE = "SECURE"
    VULNERABLE = "VULNERABLE"


class FruitType(str, Enum):
    ACHENE = "ACHENE"
    BERRY = "BERRY"
    CAPSULE = "CAPSULE"
    DRUPE = "DRUPE"
    LEGUME = "LEGUME"
    NUT = "NUT"
    MERICARP = "MERICARP"

class Measurements(BaseModel):
    "Extract units as described in passage. No conversions."
    min: Optional[float] = None
    max: Optional[float] = None
    extreme_min: Optional[float] = None
    extreme_max: Optional[float] = None
    unit: Optional[str] = None
    
class Dimensions(BaseModel):
    length: Optional[Measurements] = Field(None, description="Extract length")
    width: Optional[Measurements] = Field(None, description="Extract width")


In [ ]:
# ----------------------------
# Data Fields Models
# ----------------------------

class CoreFieldsModel(BaseModel):
    family: str = Field(..., description="Plant family name")
    genus: str = Field(..., description="Plant genus name")
    species: str = Field(..., description="Plant species epithet")
    infraspecific_epithet: Optional[str] = Field(None, description="Infraspecific taxon name following the species name")
    common_name: Optional[str] = Field(None, description="Common or vernacular name of the plant")
    hawaiian_name: Optional[List[str]] = Field(None, description="Hawaiian name of the plant, including diacritical marks")
    wagner_pg_number: Optional[str] = Field(None, description="Reference page number in Wagner's Manual of Flowering Plants of Hawaii")
    description: Optional[Description] = Field(None, description="Major plant group classification (infer from taxonomy)")
    
class OuterFlowerMorphologyModel(BaseModel):
    corolla_type: Optional[List[CorollaType]] = Field(None, description="The shape/description of petals (e.g., oblong, obovate, lanceolate)")
    corolla_color: Optional[str] = Field(None, description="Color of petals (corolla)")
    perianth_color: Optional[str] = Field(None, description="Color of perianth (undifferentiated tepals)")
    perianth_dimensions: Optional[Dimensions] = Field(None, description="Dimensions of collective outer envelope (must include petals, calyx, sepals) of flower.")
    corolla_dimensions: Optional[Dimensions] = Field(None, description="Dimensions of flower petals.")
    calyx_dimensions: Optional[Dimensions] = Field(None, description="Dimensions of flower sepals.")
    labellum_color: Optional[str] = Field(None, description="Color of labellum/lip (modified orchid petal)")
    
    labellum_dimensions: Optional[Dimensions] = Field(None, description="Dimensions of labellum/lip.")
    
    calyx_teeth_dimensions: Optional[Dimensions] = Field(None, description="Dimensions of the lobes of the calyx margin.")
    calyx_lobe_dimensions: Optional[Dimensions] = Field(None, description="Dimensions of calyx lobes in flowers with fused sepals.")
    
    upper_calyx_length: Optional[Measurements] = Field(None, description="Length of upper flower sepals specifically.")
    lower_calyx_length: Optional[Measurements] = Field(None, description="Length of lower flower sepals specifically.")
    inner_calyx_length: Optional[Measurements] = Field(None, description="Length of inner flower sepals specifically.")
    outer_calyx_length: Optional[Measurements] = Field(None, description="Length of outer flower sepals specifically.")
    
    inner_calyx_lobes_dimensions: Optional[Dimensions] = Field(None, description="Dimensions of inner calyx lobes when flowers sepals are joined together.")
    outer_calyx_lobes_dimensions: Optional[Dimensions] = Field(None, description="Dimensions of outer calyx lobes when flowers sepals are joined together.")
    
    male_calyx_dimensions: Optional[Dimensions] = Field(None, description="Dimensions of the male (staminate) flower calyx specifically.") 
    female_calyx_dimensions: Optional[Dimensions] = Field(None, description="Dimensions of the female (pistillate) flower calyx specifically.")
    
    male_calyx_lobe_dimensions: Optional[Dimensions] = Field(None, description="Dimensions of the lobes of male (staminate) flower calyx.")
    male_calyx_inner_lobe_length: Optional[Measurements] = Field(None, description="Length of the inner lobes of male (staminate) flower calyx.")
    male_calyx_outer_lobe_dimensions: Optional[Dimensions] = Field(None, description="Dimensions of the outer lobes of male (staminate) flower calyx.")
    
    female_calyx_lobe_dimensions: Optional[Dimensions] = Field(None, description="Dimensions of the lobes of female (pistillate) flower calyx.")
    female_calyx_inner_lobe_dimensions: Optional[Dimensions] = Field(None, description="Dimensions of the inner lobes of female (pistillate) flower calyx.")
    female_calyx_outer_lobe_dimensions: Optional[Dimensions] = Field(None, description="Dimensions of the outer lobes of female (pistillate) flower calyx.")
    
    calyx_tube_dimensions: Optional[Dimensions] = Field(None, description="Dimensions of the tubular, fused base of sepals (calyx tube).")
    male_calyx_tube_length: Optional[Measurements] = Field(None, description="Length of the tubular, fused base of sepals (calyx tube) on male (staminate) flowers.")
    
    corolla_tube_dimensions: Optional[Dimensions] = Field(None, description="Dimensions of the tubular, fused structure of the petals (corolla tube).")
    corolla_lobe_dimensions: Optional[Dimensions] = Field(None, description="Dimensions of the free parts of fused petals (corolla lobe).")
    corolla_lip_length: Optional[Measurements] = Field(None, description="Length of the corolla lip or labellum which is a specialized lobe of flowers with fused petals.")
    
    upper_corolla_length: Optional[Measurements] = Field(None, description="Length of upper flower petals (corolla) specifically.")
    lower_corolla_length: Optional[Measurements] = Field(None, description="Length of lower flower petals (corolla) specifically.")
    
    upper_corolla_lobes_length: Optional[Measurements] = Field(None, description="Length of upper lobes or lip of the fused petals (corolla).")
    lower_corolla_lobes_length: Optional[Measurements] = Field(None, description="Length of lower lobes or lip of the fused petals (corolla).")
    
    staminate_corolla_length: Optional[Measurements] = Field(None, description="Length of staminate flower petals (corolla) specifically.")
    pistillate_corolla_length: Optional[Measurements] = Field(None, description="Length of pistillate flower petals (corolla) specifically.")
    
    staminate_corolla_tube_dimensions: Optional[Dimensions] = Field(None, description="Dimensions of the tubular, fused structure of the petals (corolla tube) of male (staminate) flowers.")
    pistillate_corolla_tube_dimensions: Optional[Dimensions] = Field(None, description="Dimensions of the tubular, fused structure of the petals (corolla tube) of female (pistillate) flowers.")

    male_corolla_lobe_dimensions: Optional[Dimensions] = Field(None, description="Dimensions of the free parts of fused petals (corolla lobe) of male (staminate) flowers.")
    female_corolla_lobe_dimensions: Optional[Dimensions] = Field(None, description="Dimensions of the free parts of fused petals (corolla lobe) of female (pistillate) flowers.")
    
    perianth_inner_color: Optional[str] = Field(None, description="Color of inner perianth specifically.")
    perianth_outer_color: Optional[str] = Field(None, description="Color of outer perianth specifically.")
    perianth_outer_dimensions: Optional[Dimensions] = Field(None, description="Dimensions of the outer collective of the outer outer envelope of flower (must include corolla, calyx, sepals)")
    perianth_inner_dimensions: Optional[Dimensions] = Field(None, description="Dimensions of the inner collective of the outer outer envelope of flower (must include corolla, calyx, sepals)")
    perianth_tube_length: Optional[Measurements] = Field(None, description="Length of the tubular structure formed by the perianth when tepals are fused.")
    perianth_lobes_dimensions: Optional[Dimensions] = Field(None, description="Length of the lobes formed by the fusion of perianth segments.")
    
    staminate_perianth_tube_length: Optional[Measurements] = Field(None, description="Length of the tubular structure formed by the perianth when tepals are fused on staminate/male flowers.")
    pistillate_perianth_tube_length: Optional[Measurements] = Field(None, description="Length of the tubular structure formed by the perianth when tepals are fused on pistillate/female flowers.")
    
    tepal_length: Optional[Measurements] = Field(None, description="Length of tepals when petals and sepals are not differentiated.")
    staminate_tepal_length: Optional[Measurements] = Field(None, description="Length of tepals of male/staminate plants")
    pistillate_tepal_length: Optional[Measurements] = Field(None, description="Length of tepals of female/pistillate plants")
    flower_dimensions: Optional[Dimensions] = Field(None, description="Dimensions given for the overall flower.")


class FruitMorphologyModel(BaseModel):
    fruit_type: Optional[List[FruitType]] = Field(None, description="Type of  mature, ripened ovary of a flowering plant or fruit described.")
    fruit_length: Optional[Measurements] = Field(None, description="Fruit length")
    fruit_width: Optional[Measurements] = Field(None, description="Fruit width")
    fruit_diameter: Optional[Measurements] = Field(None, description="Fruit diameter")
    bur_length: Optional[Measurements] = Field(None, description="Bur (spiny fruit covering) length")
    seeds_perfruit: Optional[Measurements] = Field(None, description="Number of seeds per fruit")
    seed_length: Optional[Measurements] = Field(None, description="Seed length.")
    seed_width: Optional[Measurements] = Field(None, description="Seed width")
    seed_diameter: Optional[Measurements] = Field(None, description="Seed diameter")    
    
class InflorescenceSpecificMorphologyFieldsModel(BaseModel):
    head_length: Optional[Measurements] = Field(None, description="Capitulum length")
    head_diameter: Optional[Measurements] = Field(None, description="Capitulum diameter")
    pappus_length: Optional[Measurements] = Field(None, description="Pappus length.")
    ray_dimensions: Optional[Dimensions] = Field(None, description="Describes dimensions of rays or ray florets.")
    ray_color: Optional[str] = Field(None, description="Ray floret color")
    floret_color: Optional[str] = Field(None, description="Color of florets (must specify florets vs flower/petal color)")
    florets_length: Optional[Measurements] = Field(None, description="Length of small flowers that compose a flower head (capitulum)")
    spathe_color: Optional[str] = Field(None, description="Color of spathe")
    spathe_dimensions: Optional[Dimensions] = Field(None, description="Dimensions of spathe")
    spadix_length: Optional[Measurements] = Field(None, description="Length of spadix")

class BractInvolucreMorphologyFieldsModel(BaseModel):
    bract_dimensions: Optional[Dimensions] = Field(None, description="Bract dimensions.")
    bract_lower_length: Optional[Measurements] = Field(None, description="Lower/basal bract length")
    bract_outer_length: Optional[Measurements] = Field(None, description="Outer bract length")
    bracteoles_dimensions: Optional[Dimensions] = Field(None, description="Dimensions of bracteoles")
    involucre_dimensions: Optional[Dimensions] = Field(None, description="Involucre dimensions")    
    staminate_involucre_length: Optional[Measurements] = Field(None, description="Length of involucre of staminate/male heads.")
    pistillate_involucre_length: Optional[Measurements] = Field(None, description="Length of involucre of pistillate/female heads.")
    
class InflorescenceMorphologyModel(BaseModel):
    inflorescence_type: Optional[List[InflorescenceType]] = Field(None, description="Type of inflorescence")
    inflorescence_dimensions: Optional[Dimensions] = Field(None, description="Dimensions of the inflorescence.")
    
    peduncle_dimensions: Optional[Dimensions] = Field(None, description="Main inflorescence (peduncle) stalk dimensions")
    pedicel_dimensions: Optional[Dimensions] = Field(None, description="Individual flower stalk (pedicel) in an inflorescence dimensions")
    
    rachis_length: Optional[Measurements] = Field(None, description="Main inflorescence stalk (rachis) length")
    rachis_diameter: Optional[Measurements] = Field(None, description="Main inflorescence stalk (rachis) diameter")
    
    hypanthium_dimensions: Optional[Dimensions] = Field(None, description="Floral cup (hypanthium) dimensions")
    umbellet_length: Optional[Measurements] = Field(None, description="Secondary umbel unit length")
    
    staminate_inflorescence_dimensions: Optional[Dimensions] = Field(None, description="Male inflorescence dimensions")
    pistillate_inflorescence_dimensions: Optional[Dimensions] = Field(None, description="Female inflorescence dimensions")

    staminate_peduncle_dimensions: Optional[Dimensions] = Field(None, description="Main inflorescence (peduncle) stalk of male/staminate plant dimensions")
    pistillate_peduncle_dimensions: Optional[Dimensions] = Field(None, description="Main inflorescence (peduncle) stalk of female/pistillate plant dimensions")
    
    staminate_pedicel_dimensions: Optional[Dimensions] = Field(None, description="Individual flower stalk (pedicel) in an inflorescence of male/staminate plant dimensions")
    pistillate_pedicel_dimensions: Optional[Dimensions] = Field(None, description="Individual flower stalk (pedicel)  in an inflorescence of female/pistillate plant dimensions")

class LeafMorphologyModel(BaseModel):
    leaf_type: Optional[List[LeafType]] = Field(None, description="Leaf structure type (simple or compound)")
    leaf_shape_type: Optional[List[LeafShapeType]] = Field(None, description="Shape of leaf blade (ovate, lanceolate, cordate, etc.)")
    leaf_margin_type: Optional[List[LeafMarginType]] = Field(None, description="Type of leaf edge (entire, serrate, lobed, etc.)")
    phyllotaxy_type: Optional[List[PhyllotaxyType]] = Field(None, description="The arrangement of leaves around the stem (words like pairs and leaflet information are important indicators).")

    leaf_dimensions: Optional[Dimensions] = Field(None, description="Dimensions of leaves or individual leaf blade")
    petiole_length: Optional[Measurements] = Field(None, description="Length of leaf stalk (petiole).")

class LeafIndumentumModel(BaseModel):
    leaf_hair_type: Optional[List[LeafHairType]] = Field(None, description="Type of leaf surface indumentum.")
    upper_leaf_hair_type: Optional[List[LeafHairType]] = Field(None, description="Specific description of upper leaf surface indumentum.")
    lower_leaf_hair_type: Optional[List[LeafHairType]] = Field(None, description="Specific description of lower leaf surface indumentum.")

class LeafletMorphologyFieldsModel(BaseModel):
    leaflets_leaf_type: Optional[List[LeafType]] = Field(None, description="Leaflet leaf structure type in compound leaves")
    leaflets_leaf_shape_type: Optional[List[LeafShapeType]] = Field(None, description="Shape of individual leaflets")
    leaflets_leaf_margin_type: Optional[List[LeafMarginType]] = Field(None, description="Margin type of leaflets")
    leaflets_leaf_dimensions: Optional[Dimensions] = Field(None, description="Dimensions of individual leaflets")
    
class JuvenileLeafFieldsModel(BaseModel):
    juvenile_leaf_type: Optional[List[LeafType]] = Field(None, description="Leaf structure type in juvenile plants")
    juvenile_leaf_shape_type: Optional[List[LeafShapeType]] = Field(None, description="Shape of juvenile leaves")
    juvenile_leaf_margin_type: Optional[List[LeafMarginType]] = Field(None, description="Margin type of juvenile leaves")
    juvenile_leaf_dimensions: Optional[Dimensions] = Field(None, description="Dimensions of juvenile leaves")
    juvenile_leaf_hair_type: Optional[List[LeafHairType]] = Field(None, description="Indumentum type on juvenile leaves")

class StemMorphologyModel(BaseModel):
    stem_height: Optional[Measurements] = Field(None, description="Overall height of the plant or length of the main stem")
    stem_hair_type: Optional[List[StemHairType]] = Field(None, description="Describes the stem hair type of the main stem (NOT penduncle) (e.g., glabrous, hirsute, tomentose, etc.)")
    
class LifeFormModel(BaseModel):
    life_form_type: Optional[List[LifeFormType]] = Field(None, description="Growth habit or life form (annual herb, perennial herb, shrub, tree, vine, etc.)")
    breeding_type: Optional[List[BreedingType]] = Field(None, description="Plant reproductive class (infer from taxonomy)")

class ReproductiveMorphologyModel(BaseModel):
    ploidy: Optional[list] = Field(None, description="Ploidy level for the specific species expressed as a function of n (e.g., 1n, 2n or 3n, etc..)")
    chromosome_number: Optional[list] = Field(None, description="Species specific chromosome count (integer number)")
    
class DistributionFieldsModel(BaseModel):
    island_type: Optional[List[Location]] = Field(None, description= "Hawaiian islands the plant is found.")
    origin: Optional[List[OriginType]] = Field(None, description="Origin abbreviated at the beginning of the passage (end/endemic, nat/naturalized, PC/paleotropical cosmopoltian.)")
    status: Optional[List[StatusType]] = Field(None, description="Conservation or rarity status")
    

In [ ]:
# ----------------------------
# Flag models
# ----------------------------

def FlagField(desc: str):
    return Field(default_factory=Flag, description=desc)

class Flag(BaseModel):
    value: bool = Field(False, description="Set True if the passage clearly provides specific information for the given field.")
    description: str = Field("", description="Short justification or quote to justify the flag or 'not mentioned'.")
    
def to_bools(self) -> Dict[str, bool]:
    """Flatten to {flag_name: bool}."""
    return {k: getattr(self, k).value for k in self.model_fields}

def reasons(self) -> Dict[str, str]:
    """Flatten to {flag_name: description}."""
    return {k: getattr(self, k).description for k in self.model_fields}

In [ ]:
# ----------------------------
# GroupFlags Model -> Describes data fields to be flagged
# ----------------------------

class GroupFlags(BaseModel):
    has_core:             Flag = FlagField("Contains core taxonomic/identification metadata.")
    has_life_form: Flag = FlagField("Describes life form such as annual herb, shrub, tree, vine, etc.")
    has_outer_flower_morphology: Flag = FlagField("Describes outer flower morphology including petals, corolla, calyx, perianth, or sepal morphology.")
    has_fruit_morphology: Flag = FlagField("Describes fruit morphology such as fruit type, size, or color.")
    has_inflorescence_specific_morphology: Flag = FlagField("Describes or mentions head, pappus, spadix, spathe, ray, rays, or floret morphology.")
    has_bract_involucre_morphology: Flag = FlagField("Describes involucre, bract or bracteole morphology.")
    has_inflorescence_morphology: Flag = FlagField("Describes inflorescence morphology such as type of flower cluster arrangement (cyme, raceme, etc.) or dimensions of inflorescence parts.")
    has_leaf_morphology: Flag = FlagField("Describes leaf morphology such as leaf type, shape, margin, or phyllotaxy.")
    has_leaflet_morphology: Flag = FlagField("Specifically mentions leaflet morphology.")
    has_juvenile_leaf_morphology: Flag = FlagField("Specifically mentions juvenile leaf morphology.")
    has_stem_morphology: Flag = FlagField("Describes stem morphology such as plant height or hair type.")
    has_reproductive_morphology: Flag = FlagField("Describes reproductive morphology such as life form, breeding type, flower dimensions, or cytology.")
    has_distribution: Flag = FlagField("Describes plant distribution such as specific Hawaiian islands, origin status, or conservation status.")
    has_leaf_indumentum: Flag = FlagField("Describes leaf indumentum such as hair type of leaf surface.")
    
# ----------------------------
# FLAG_TO_MODEL dict -> Maps flags to models to data fields
# ----------------------------
    
FLAG_TO_MODEL: Dict[str, tuple] = {
    "has_core":                              ("core",                             CoreFieldsModel),
    "has_life_form":                             ("life_form",                             LifeFormModel),
    "has_outer_flower_morphology":                              ("outer_flower_morphology",                             OuterFlowerMorphologyModel),
    "has_fruit_morphology":                             ("fruit_morphology",                             FruitMorphologyModel),
    "has_inflorescence_specific_morphology":                             ("Inflorescence_Specific",                             InflorescenceSpecificMorphologyFieldsModel),
    "has_bract_involucre_morphology":                             ("bract_involucre_morphology",                             BractInvolucreMorphologyFieldsModel),
    "has_inflorescence_morphology":                             ("inflorescence_morphology",                             InflorescenceMorphologyModel),
    "has_leaf_morphology":                             ("leaf_morphology",                             LeafMorphologyModel),
    "has_leaflet_morphology":                             ("leaflet_morphology",                             LeafletMorphologyFieldsModel),
    "has_juvenile_leaf_morphology":                             ("juvenile_leaf_morphology",                             JuvenileLeafFieldsModel),
    "has_stem_morphology":                             ("stem_morphology",                             StemMorphologyModel),
    "has_reproductive_morphology":                             ("reproductive_morphology",                             ReproductiveMorphologyModel),
    "has_distribution":                             ("distribution",                             DistributionFieldsModel),
    "has_leaf_indumentum":                             ("leaf_indumentum",                             LeafIndumentumModel),
    }

In [ ]:
# ----------------------------
# Generic agent runner
# ----------------------------
async def _run_agent(result_type: type, system_prompt: str, sample: str, model) -> Any:
    agent = Agent(
        model=model,
        output_type=result_type,
        system_prompt=system_prompt,
    )
    resp = await agent.run(sample)
    return resp.output


# ----------------------------
# Orchestrator
# ----------------------------
async def extract_plant(sample: str, model) -> Dict[str, Any]:
    # Phase 1: detect all flags in parallel
    flag_names = list(GroupFlags.model_fields)
    flag_results = await asyncio.gather(*[
        _run_agent(
            Flag,
            f"Botanical specialist. Does the passage contain: {GroupFlags.model_fields[name].description} "
            "Set value=True only if clearly supported. Quote the relevant text.",
            sample, model,
        )
        for name in flag_names
    ], return_exceptions=True)

    flag_values: Dict[str, Flag] = {}
    for name, result in zip(flag_names, flag_results):
        if isinstance(result, Exception):
            print(f"Flag error [{name}]: {result}")
            flag_values[name] = Flag()
        else:
            flag_values[name] = result
    flags = GroupFlags(**flag_values)

    # Phase 2: extract all flagged groups in parallel (deduplicate shared models)
    seen: set = set()
    active: list[tuple[str, type, str]] = [
        (group_name, model_cls, getattr(flags, flag_name).description)
        for flag_name, (group_name, model_cls) in FLAG_TO_MODEL.items()
        if getattr(flags, flag_name).value and group_name not in seen and not seen.add(group_name)
    ]
    active_groups: Dict[str, type] = {group_name: model_cls for group_name, model_cls, _ in active}

    group_results = await asyncio.gather(*[
        _run_agent(
            model_cls,
            f"You are a botantical expert. Extract the data fields provided, in the text passage. Return null for fields that are not clearly stated.",
            sample, model,
        )
        for group_name, model_cls, flag_desc in active
    ], return_exceptions=True)

    results = {
        group_name: data
        for (group_name, _, _), data in zip(active, group_results)
        if not isinstance(data, Exception)
    }

    # Phase 3: validate and self-correct
    results = await validate_and_correct(sample, model, results, active_groups)

    return results, flags

In [ ]:
## Skip validation for data fields that are unlikely to hallucinate (saves tokens, reduces chance for validation hallucination)
SKIP_VALIDATION = {"core"}

# ----------------------------
# Validation models
# ----------------------------
class FieldFlag(BaseModel):
    field: str = Field(..., description= "Field name")
    reason: str = Field(..., description= "Reasoning field was flagged (hallucination or constraint violation)")

# List-based so the LLM doesn't have to return a dict (avoids ValidationError)
class GroupValidationItem(BaseModel):
    group_name: str
    is_valid: bool = Field(default=True, description="False if the values are clearly hallucinated or violate model constraints.")
    flagged_fields: List[FieldFlag] = Field(default_factory=list)

class BatchValidation(BaseModel):
    groups: List[GroupValidationItem]

# ----------------------------
# Extract species-level text (trim family/genus preamble for correction calls)
# ----------------------------
def extract_species_text(text: str) -> str:
    """Return species-level section only; fall back to full text if not found."""
    import re
    match = re.search(r'\nSpecies:.*', text, re.DOTALL | re.IGNORECASE)
    if match:
        return match.group(0).strip()
    cutoff = int(len(text) * 0.4)
    return text[-cutoff:] if cutoff < len(text) else text


# ----------------------------
# Parallel group re-extractor (used by validate_and_correct for corrections)
# ----------------------------
async def parse_all_groups_parallel(
    sample: str,
    model,
    group_models: Dict,
    group_hints: Dict[str, str] = None,
) -> Dict:
    group_hints = group_hints or {}

    async def run_one(name, cls):
        hint = group_hints.get(name, "")
        prompt = f"Botanical specialist. Extract only {name} fields. Return null for unsupported fields."
        if hint:
            prompt += f"\n{hint}"
        return name, await _run_agent(cls, prompt, sample, model)

    pairs = await asyncio.gather(*[run_one(n, c) for n, c in group_models.items()], return_exceptions=True)
    return {n: d for item in pairs if not isinstance(item, Exception) for n, d in [item]}


# ----------------------------
# Batch validate all groups in ONE call
# ----------------------------
async def validate_all_batch(sample: str, model, results: Dict) -> Dict[str, GroupValidationItem]:
    to_validate = {n: r for n, r in results.items() if n not in SKIP_VALIDATION}
    if not to_validate:
        return {}

    val_prompt = (
        "You are a botanical data quality auditor. "
        "Given source text and extracted data for multiple field groups, "
        "flag any field that:\n"
        "- contradicts the source text (hallucination)\n"
        "- is not specific to the data field or species being extracted.\n"
        "Leave is_valid=True and flagged_fields=[] if all fields are supported by the source text."
    )

    groups_json = json.dumps(
        {name: json.loads(r.model_dump_json()) for name, r in to_validate.items()},
        indent=2,
    )

    agent = Agent(
        model=model,
        output_type=BatchValidation,
        system_prompt=val_prompt,
    )
    resp = await agent.run(f"SOURCE TEXT:\n{sample}\n\nEXTRACTED DATA:\n{groups_json}")

    returned = {item.group_name: item for item in resp.output.groups}

    # Groups the LLM didn't mention are implicitly valid
    return {
        name: returned.get(name, GroupValidationItem(group_name=name))
        for name in to_validate
    }


# ----------------------------
# Validate + self-correct loop
# ----------------------------
async def validate_and_correct(
    sample: str,
    model,
    results: Dict,
    active_groups: Dict,
    group_hints: Dict[str, str] = None,
    max_retries: int = 2,
) -> Dict:
    current = dict(results)
    group_hints = group_hints or {}
    correction_text = sample

    for attempt in range(max_retries):
        validations = await validate_all_batch(sample, model, current)
        flagged = {n: v for n, v in validations.items() if not v.is_valid}

        if not flagged:
            print(f"All groups valid" + (f" after {attempt} correction(s)." if attempt else "."))
            return current

        print(f"Attempt {attempt + 1}/{max_retries}: correcting {list(flagged.keys())}")
        for name, val in flagged.items():
            for f in val.flagged_fields:
                print(f"  {name}.{f.field}: {f.reason}")

        correction_hints = {}
        for name in flagged:
            issues = "; ".join(f"{f.field}: {f.reason}" for f in flagged[name].flagged_fields)
            base = group_hints.get(name, "")
            correction_hints[name] = f"{base}\nCorrection notes: {issues}".strip()

        flagged_models = {n: active_groups[n] for n in flagged if n in active_groups}
        corrections = await parse_all_groups_parallel(correction_text, model, flagged_models, group_hints=correction_hints)
        current.update(corrections)

    return current

In [ ]:
# ----------------------------
# Run Individual Wagner Extraction -> Apiaceae Coriandrum sativum
# ----------------------------

### res = raw results from extraction agent (dict)
### flags = results of fields flagged (__main__.GroupFlags)
### __main__.GroupFlags = GroupFlags(data_field = Flag(value = True/False, description = reasoning for flag value))

# pd.set_option('display.max_columns', None)
# pd.set_option('display.max_rows', None)

res, flags = await extract_plant(text_inputs['Coriandrum_sativum'.upper()], model)

## Formats results to final dataframe
results_df = results_to_df(res)
results_df

In [ ]:
# ----------------------------
# Run Wagner Extraction on all input text -> 13 species passages
# ----------------------------

output_path = "./extracted_wagner_data.csv"

dfs = []

for i in text_inputs.keys():
    print(f"Processing {i}...")
    res, flags = await extract_plant(text_inputs[i], model)
    
    species_key = f"{res['core'].family}_{res['core'].genus}_{res['core'].species}".lower()
    
    ## Adds manually extracted data to the final output dataframe (for easy comparison)
    man_row = manual_extracted_data_df[manual_extracted_data_df['species_key'] == species_key]
    auto_df = results_to_df(res)
        
    dfs.append(auto_df)
    dfs.append(man_row)
    # break

# Combine all results and output to CSV
if dfs:
    final_df = pd.concat(dfs, ignore_index=True)
    final_df.to_csv(f"{output_path}", index=False)
    print(f"Saved to {output_path}")